# Lab Assignment 11: Coreference Resolution & Question Answering

**Objective:**
In this lab you will implement and evaluate two core NLP tasks:
(A) **Coreference Resolution** using a SpanBERT-based model, and
(B) **Question Answering** by fine-tuning BERT on the SQuAD reading comprehension dataset.
You will also explore the connection between the two tasks.

**Task:** (A) Coreference chain detection and evaluation. (B) Span extraction from a passage given a question.

**Learning outcomes:**
1. Understand and use a SpanBERT-based coreference model.
2. Implement B-cubed evaluation for coreference.
3. Fine-tune (or use a fine-tuned) BERT model for span-based QA.
4. Implement SQuAD-style EM and F1 evaluation.
5. Explore the Open-Domain QA retriever-reader pipeline.

| Part | Topic | Points |
|---|---|---|
| 1 | Setup & Data | 5 |
| 2 | Coreference: SpanBERT Model | 20 |
| 3 | Coreference: Manual Analysis & Evaluation | 20 |
| 4 | Question Answering: BERT for SQuAD | 25 |
| 5 | QA Evaluation: EM and F1 | 15 |
| 6 | Open-Domain QA Pipeline | 10 |
| 7 | Connecting Coref and QA | 5 |
| | **Total** | **100** |

**Submission:** Complete all `# TODO` sections and all markdown answers. Run **Kernel > Restart & Run All** before submitting.


---
## Theory Recap

**Coreference Resolution** groups all text spans (mentions) that refer to the same real-world entity.
Three types of mentions: *pronouns*, *named entities*, and *noun phrases*.

**Mention-Ranking** scores each mention against all preceding candidates:
$$s(i,j) = s_m(i) + s_m(j) + s_a(i,j)$$
A dummy antecedent ε allows the model to start a new cluster.

**SpanBERT** represents span (i,j) as: $g(i,j) = [h_i, h_j, \tilde{h}(i,j), \phi(i,j)]$

**BERT for QA (SQuAD):** Fine-tune with two linear classifiers on the final hidden states:
$$P_{start}(i) = \text{softmax}(W_{start}\, h_i), \quad P_{end}(i) = \text{softmax}(W_{end}\, h_i)$$
The predicted answer is $P[\text{argmax}(P_{start}) : \text{argmax}(P_{end})+1]$.


**B-cubed Precision/Recall** for mention $m$:
$$P_m = \frac{|C_{gold}(m) \cap C_{sys}(m)|}{|C_{sys}(m)|}, \quad R_m = \frac{|C_{gold}(m) \cap C_{sys}(m)|}{|C_{gold}(m)|}$$


---
## Part 1: Setup and Data (5 points)

### 1.1 Install and Import Libraries


In [1]:
# TODO: install required libraries
# !pip install -q fastcoref transformers datasets evaluate scikit-learn
# !python -m spacy download en_core_web_sm -q


In [2]:
# TODO: import torch, transformers (pipeline, AutoTokenizer, AutoModelForQuestionAnswering),
# fastcoref (FCoref), datasets (load_dataset), re, string, collections (Counter), matplotlib
# Set random seeds for reproducibility.


### 1.2 Test Texts for Coreference
Use the following texts for Parts 2 and 3. **Do not change them.**


In [3]:
TEXT_1 = (
    'Akhila met Prajwal at the local park. '
    "Akhila's son Akash was just two months younger than her son Prajwal. "
    'She resigned herself to make Akash the best tree in the play. '
    'She bought him a brown T-shirt to represent the trunk.'
)

TEXT_2 = (
    'Barack Obama was born in Hawaii. '
    'He served as the 44th President of the United States. '
    'Obama attended Columbia University. '
    'His presidency lasted from 2009 to 2017.'
)

TEXT_3 = (
    'Maria Curie was a Polish-born physicist and chemist. '
    'She was the first woman to win a Nobel Prize. '
    'In 1906 she became the first woman professor at the University of Paris. '
    'Curie discovered polonium and radium with her husband Pierre.'
)

print('Sample texts loaded.')

Sample texts loaded.


---
## Part 2: Coreference Resolution using SpanBERT (20 points)

### 2.1 Load the Model and Run Prediction
Load the `FCoref` model and run coreference resolution on **TEXT_1**, **TEXT_2**, and **TEXT_3**.
Print the coreference clusters (as strings) for each text.


In [4]:
# TODO: Load FCoref and run on the three texts
# model = FCoref(device='cpu')
# preds = model.predict(texts=[TEXT_1, TEXT_2, TEXT_3])
# for i, (text, pred) in enumerate(zip([TEXT_1, TEXT_2, TEXT_3], preds)):
#     print(f'--- Text {i+1} ---')
#     for cluster in pred.get_clusters(as_strings=True):
#         print(' → '.join(cluster))
#     print()


### 2.2 Markdown Questions
Answer in the cell below:
1. For TEXT_1, list all the coreference clusters the model found. Are they correct?
2. Which mentions were **missed** (false negatives) and which were **wrong** (false positives)?
3. Why might the model confuse 'her son' in TEXT_1? What linguistic phenomenon makes this hard?


**Your answers:**

1. *(your answer)*

2. *(your answer)*

3. *(your answer)*


### 2.3 Mention Detection: Types and Counts
Write a function `count_mention_types(text)` that uses spaCy to count the number of:
- Pronouns (using POS tags)
- Named entities (using NER)
- Noun phrases (using noun chunks)
Apply it to all three texts. Print results in a table.


In [5]:
import spacy
nlp = spacy.load('en_core_web_sm')

def count_mention_types(text):
    """
    Returns a dict: {'pronouns': int, 'named_entities': int, 'noun_phrases': int}
    """
    # TODO
    pass

for i, text in enumerate([TEXT_1, TEXT_2, TEXT_3], 1):
    counts = count_mention_types(text)
    print(f'Text {i}: {counts}')


/data3/chhamman/miniconda3/lib/python3.14/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12030). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Text 1: None
Text 2: None
Text 3: None


---
## Part 3: Manual Evaluation with B-cubed (20 points)

### 3.1 Gold Standard Clusters (TEXT_2 Only)
For TEXT_2, define the **gold standard** coreference clusters using the mention offsets from the text.
The character-level mentions are given — use them as entity labels (strings).


In [6]:
# Gold clusters for TEXT_2 (mention strings as identifiers)
GOLD_TEXT2 = [
    {'Barack Obama', 'He', 'Obama', 'His'},  # Cluster 1: the person
    {'the United States'},                    # Cluster 2: the country
    {'Columbia University'},                  # Cluster 3: the university
    {'Hawaii'},                               # Singleton: Hawaii
]
# Note: singletons are valid gold clusters too
print('Gold clusters defined:', len(GOLD_TEXT2), 'clusters')

Gold clusters defined: 4 clusters


### 3.2 Extract System Clusters for TEXT_2
Extract the coreference clusters that the model predicted for TEXT_2 (from Part 2.1).
Convert them to sets of mention strings (as above) for comparison.


In [7]:
# TODO: Extract system clusters for TEXT_2 as a list of sets of mention strings
# system_clusters_text2 = [...]
# print(system_clusters_text2)


### 3.3 Implement and Compute B-cubed
Implement the `bcubed(gold_clusters, system_clusters)` function and compute B-cubed P, R, F1 for TEXT_2.


In [8]:
def bcubed(gold_clusters, system_clusters):
    """
    Compute B-cubed Precision, Recall, F1.
    Arguments:
        gold_clusters:   list of sets of mention strings (gold standard)
        system_clusters: list of sets of mention strings (system output)
    Returns: (precision, recall, f1) — all floats rounded to 3 decimal places

    Algorithm:
    1. Build a mention → cluster_index lookup for gold and system.
    2. For each mention m:
         overlap = |gold_cluster(m) ∩ system_cluster(m)|
         P_m = overlap / |system_cluster(m)|
         R_m = overlap / |gold_cluster(m)|
    3. Average P_m and R_m across all mentions.
    4. Compute F1 from averaged P and R.
    """
    # TODO
    pass

# Compute B-cubed for TEXT_2
# P, R, F1 = bcubed(GOLD_TEXT2, system_clusters_text2)
# print(f'B-cubed for TEXT_2: P={P}, R={R}, F1={F1}')


### 3.4 Short Questions
1. What does it mean when B-cubed Precision is 1.0 but Recall is low?
2. What does it mean when B-cubed Recall is 1.0 but Precision is low?
3. Besides B-cubed, name two other standard coreference evaluation metrics used for OntoNotes.


**Your answers:**

1. *(your answer)*

2. *(your answer)*

3. *(your answer)*


---
## Part 4: Question Answering with BERT (25 points)

### 4.1 Setup the QA Model
Load `deepset/bert-base-cased-squad2` using HuggingFace `AutoTokenizer` and `AutoModelForQuestionAnswering`.


In [9]:
# TODO: load tokenizer and model
# model_name = 'deepset/bert-base-cased-squad2'
# tokenizer  = AutoTokenizer.from_pretrained(model_name)
# qa_model   = AutoModelForQuestionAnswering.from_pretrained(model_name)
# qa_model.eval()


### 4.2 Implement Span Extraction
Implement `extract_answer(question, context, tokenizer, model)` that:
1. Tokenizes the (question, context) pair.
2. Runs the model.
3. Finds the best valid span: `start <= end` and both within the context tokens.
4. Returns the answer string and its score (start_logit[s] + end_logit[e]).


In [10]:
import torch

def extract_answer(question, context, tokenizer, model, max_answer_len=50):
    """
    Extract answer span from context given a question.
    Arguments:
        question:       str
        context:        str
        tokenizer:      HuggingFace tokenizer
        model:          AutoModelForQuestionAnswering
        max_answer_len: maximum number of tokens in the answer span
    Returns:
        answer: str — the extracted answer text
        score:  float — start_logit[s] + end_logit[e]
    Requirements:
      - Must enforce end >= start
      - Must enforce end - start < max_answer_len
      - Must use torch.no_grad() for inference
      - Must handle the [CLS] and [SEP] tokens correctly
        (answer must come from the context portion, not the question)
    """
    # TODO
    pass


### 4.3 Test on SQuAD-Style Examples
Test your function on the following (passage, question, gold_answer) triples. Print the predicted answer and score.


In [11]:
squad_examples = [
    {
        'context':  'The Tesla family moved to Gospic in 1862, where Tesla father worked as a pastor. '
                    'Nikola completed Primary School and then the Lower Real Gymnasium.',
        'question': 'Where did the Tesla family move in 1862?',
        'gold':     'Gospic'
    },
    {
        'context':  'In 1861, Tesla attended the Primary School in Smiljan '
                    'where he studied German, arithmetic, and religion.',
        'question': 'What subjects did Tesla study at Primary School?',
        'gold':     'German, arithmetic, and religion'
    },
    {
        'context':  'Kannada language is the official language of Karnataka and '
                    'spoken as a native language by about 66.54 percent of the people as of 2011. '
                    'Other minorities were Urdu (10.83 percent) and Telugu (5.84 percent).',
        'question': 'What percentage of people speak Kannada natively?',
        'gold':     '66.54 percent'
    },
]

for ex in squad_examples:
    # TODO: call extract_answer and print result
    pass


### 4.4 Visualise Start and End Logits
For the first example in `squad_examples`, plot a bar chart of the start logits and end logits (one plot each), with a vertical line marking the predicted start/end position. Label the x-axis with token strings.


In [12]:
# TODO: tokenize the first example, get logits, plot two bar charts
# (start logits and end logits), marking the predicted positions


### 4.5 Unanswerable Questions (SQuAD v2.0)
The model `deepset/bert-base-cased-squad2` was trained on SQuAD v2.0 which includes unanswerable questions. Run the model on the context and question below. Look at the score — what does a low/high score mean?


In [13]:
context_v2  = 'Tesla attended school in Smiljan where he studied German and arithmetic.'
question_v2 = 'What year was Tesla born?'

# TODO: call extract_answer and print the answer and score
# Compare the score to the answerable examples above.
# In a markdown cell, explain what SQuAD v2.0 adds over v1.1 and how models handle it.


**Your answer:** *(Explain the difference between SQuAD v1.1 and v2.0 and how models detect unanswerable questions)*


---
## Part 5: QA Evaluation — EM and F1 (15 points)

### 5.1 Implement Normalise, EM and Token F1


In [14]:
import re, string
from collections import Counter

def normalize_answer(text):
    """
    Lowercase, remove articles (a, an, the), remove punctuation, fix whitespace.
    """
    # TODO
    pass

def exact_match(prediction, gold_answers):
    """
    Return 1 if prediction exactly matches any gold answer after normalisation, else 0.
    gold_answers is a list of strings.
    """
    # TODO
    pass

def token_f1(prediction, gold_answers):
    """
    Return the maximum token-level F1 over all gold answers.
    Compute precision and recall on the BAG OF WORDS of the normalised strings.
    """
    # TODO
    pass


### 5.2 Evaluate on the SQuAD Examples
For each example in `squad_examples`, compute EM and F1 between your predicted answer and the gold answer. Print a table of results and average EM and F1.


In [15]:
# TODO: run extract_answer on each example, compute EM and F1, print a table
# Print: average EM and average F1 across all examples


### 5.3 Analysis
For any example where EM=0 but F1>0, explain in a markdown cell **why** EM is stricter than F1 and whether the model's answer is actually reasonable.


**Your analysis:**

*(your answer)*


---
## Part 6: Open-Domain QA Pipeline (10 points)

In reading comprehension, the passage is given. In open-domain QA, we must first **retrieve** relevant passages.

### 6.1 Implement a Simple BM25-style Retriever
Given a small corpus of documents and a query, implement `bm25_retrieve(query, corpus, k=3)` that returns the top-k documents using **TF-IDF** scoring (using `sklearn.feature_extraction.text.TfidfVectorizer`).


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

CORPUS = [
    'Barack Obama was born in Hawaii and served as the 44th US President from 2009 to 2017.',
    'Nikola Tesla was a Serbian-American inventor known for his work on alternating current electricity.',
    'Marie Curie discovered polonium and radium and was the first woman to win the Nobel Prize.',
    'The Eiffel Tower is located in Paris and was constructed between 1887 and 1889.',
    'Python is a high-level programming language created by Guido van Rossum in 1991.',
    'The Amazon rainforest produces 20 percent of the worlds oxygen and spans 9 countries.',
    'Albert Einstein developed the theory of relativity and won the 1921 Nobel Prize in Physics.',
    'The Great Wall of China was built over many centuries to protect Chinese states from invasions.',
]

def bm25_retrieve(query, corpus, k=3):
    """
    Retrieve top-k documents most relevant to the query using TF-IDF.
    Arguments:
        query:  str
        corpus: list of str
        k:      int — number of documents to return
    Returns:
        list of (score, document_string), sorted by score descending
    """
    # TODO: use TfidfVectorizer to vectorize corpus + query
    # compute cosine similarity between query vector and corpus vectors
    # return top-k (score, doc) pairs
    pass


### 6.2 Build the Full Retriever-Reader Pipeline
Implement `open_domain_qa(question, corpus, retriever, reader_fn, k=3)` that:
1. Retrieves the top-k passages using `retriever`.
2. Runs `reader_fn` (your `extract_answer`) on each passage.
3. Returns the answer with the highest score.


In [17]:
def open_domain_qa(question, corpus, retriever, reader_fn, k=3):
    """
    Arguments:
        question:   str
        corpus:     list of str
        retriever:  callable(query, corpus, k) → list of (score, doc)
        reader_fn:  callable(question, context) → (answer, score)
        k:          number of passages to retrieve
    Returns:
        best_answer: str
        best_score:  float
    """
    # TODO
    pass

# Test it
test_questions = [
    'Where was Barack Obama born?',
    'Who discovered radium?',
    'When was the Eiffel Tower built?',
]
for q in test_questions:
    # TODO: call open_domain_qa and print results
    pass


### 6.3 Short Answer
Explain in 3–4 lines why **dense retrieval (DPR)** can find relevant passages that BM25 might miss. Give a concrete example.


**Your answer:**

*(your answer)*


---
## Part 7: Connecting Coreference and Question Answering (5 points)

### 7.1 Coref-Aided QA
Consider the passage below where coreference is **required** to answer the question correctly.


In [18]:
coref_passage = (
    'Maria was the top student in her class. '
    'She won the national science prize in 2023. '
    'The award was presented to her by the Prime Minister.'
)
coref_question = 'Who won the national science prize?'

# Step 1: Run coreference resolution on the passage
# TODO: use FCoref to get clusters

# Step 2: Run QA directly on the passage
# TODO: run extract_answer — what does it return?

# Step 3: Show how resolving 'She' → 'Maria' helps answer correctly
resolved_passage = coref_passage.replace('She won', 'Maria won').replace(' her by', ' Maria by')
# TODO: run extract_answer on resolved_passage — compare results


### 7.2 Reflection
In 4–5 sentences, explain:
1. Why coreference resolution can improve QA performance.
2. Give a type of question where coref is **essential** (not just helpful).
3. What challenges remain even after combining coref + QA?


**Your reflection:**

*(your answer)*
